# Operations Research 3: Examples for Theory

This code is for students who take the class Operations Research. Students should finish the installation of Gurobi and Python before workersed and make sure an academic liscense for Gurobi is applied and activated.

We introduce an example for Network flow problem in order to help students understand how to implement theories introduced in lectures with codes. More instruction is provided in the lecture video.

In [1]:
% autoreload
%autoreloload_extad 2

UsageError: Line magic function `%` not found.


# Network Flow Problem
## Example: Assignment Problem

Import instance of an assignment problem. \
In the file, we have the arcs of a bipartite graph. Each arc means if the job $i$ is assigned to worker $j$, we should pay the cost $c_{ij}$. \
Each worker and job should be assigned no greater than one time in an assignment problem.

In [2]:
from gurobipy import *
import pandas as pd
import numpy as np

from op_research.config import DATA_DIR, SOLVER

2026-06-24 12:40:30.009 | INFO     | op_research.config:<module>:9 - ROOT_DIR: /Users/mmh54/Documents/github/op-research-algorithms
2026-06-24 12:40:30.010 | INFO     | op_research.config:<module>:10 - DATA_DIR: /Users/mmh54/Documents/github/op-research-algorithms/data


In [5]:
graph_info = pd.read_excel(DATA_DIR / 'bipartite_arcs_set.xlsx', 'graph')
print(graph_info)

   jobs workers  cost
0     A       E     2
1     A       F  9999
2     A       G    10
3     A       H     7
4     B       E  9999
5     B       F     4
6     B       G     3
7     B       H  9999
8     C       E     8
9     C       F     2
10    C       G     5
11    C       H  9999
12    D       E     7
13    D       F  9999
14    D       G     1
15    D       H     6


Prepare data into different format to make us construct model more efficiently.

In [6]:
# put arcs into a dictionary
flow_dict = {}

for i in range(len(graph_info)):
    flow_dict[(graph_info['jobs'][i], graph_info['workers'][i])] = graph_info['cost'][i]
    
flow_dict

{('A', 'E'): np.int64(2),
 ('A', 'F'): np.int64(9999),
 ('A', 'G'): np.int64(10),
 ('A', 'H'): np.int64(7),
 ('B', 'E'): np.int64(9999),
 ('B', 'F'): np.int64(4),
 ('B', 'G'): np.int64(3),
 ('B', 'H'): np.int64(9999),
 ('C', 'E'): np.int64(8),
 ('C', 'F'): np.int64(2),
 ('C', 'G'): np.int64(5),
 ('C', 'H'): np.int64(9999),
 ('D', 'E'): np.int64(7),
 ('D', 'F'): np.int64(9999),
 ('D', 'G'): np.int64(1),
 ('D', 'H'): np.int64(6)}

In [7]:
# get workers and jobs sets
workers_spot = set(np.unique(graph_info['workers']))
jobs_spot = set(np.unique(graph_info['jobs']))

In [9]:
# seperate arcs and cost
arcs, cost = multidict(flow_dict)

In [11]:
cost

{('A', 'E'): np.int64(2),
 ('A', 'F'): np.int64(9999),
 ('A', 'G'): np.int64(10),
 ('A', 'H'): np.int64(7),
 ('B', 'E'): np.int64(9999),
 ('B', 'F'): np.int64(4),
 ('B', 'G'): np.int64(3),
 ('B', 'H'): np.int64(9999),
 ('C', 'E'): np.int64(8),
 ('C', 'F'): np.int64(2),
 ('C', 'G'): np.int64(5),
 ('C', 'H'): np.int64(9999),
 ('D', 'E'): np.int64(7),
 ('D', 'F'): np.int64(9999),
 ('D', 'G'): np.int64(1),
 ('D', 'H'): np.int64(6)}

After preparing data, we construct a model to solve the assignment problem.

In [ ]:
eg3 = Model('Assignment')

# add variables (arcs)
x = {}
for arc in arcs:
    # Because of the Total Unimodularity property, the continuous relaxation will produce integer solutions
    x[arc] = eg3.addVar(lb = 0, vtype = GRB.CONTINUOUS, name = 'x' + arc[0] + arc[1]) 
# set objective value
eg3.setObjective(quicksum(x[arc] * cost[arc] for arc in arcs), GRB.MINIMIZE)

# each worker and job could be select no less than one time
for job in jobs_spot:
    arcs_list = []
    for arc in arcs:
        if arc[0] == job:
            arcs_list.append(x[arc])
        else:
            pass   
    eg3.addConstr(quicksum(arcs_list) == 1, name = 'job' + job)

for worker in workers_spot:
    arcs_list = []
    for arc in arcs:
        if arc[1] == worker:
            arcs_list.append(x[arc])
        else:
            pass
    eg3.addConstr(quicksum(arcs_list) == 1, name = 'worker' + worker)

eg3.optimize()

# Print solution
for var in eg3.getVars():
    print(var.varName, '=', var.x)
print("objective value =", eg3.objVal)

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F80)

CPU model: Apple M5
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 8 rows, 16 columns and 32 nonzeros (Min)
Model fingerprint: 0xfedac085
Model has 16 linear objective coefficients
Variable types: 0 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 20008.000000
Presolve time: 0.00s
Presolved: 8 rows, 16 columns, 32 nonzeros
Variable types: 0 continuous, 16 integer (16 binary)

Root relaxation: objective 1.300000e+01, 6 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0      13.0000000   13.00000  0.0